# 내적과 코사인 유사도: 크기 변화와 검색 순위

- 기준 TIL: [2026-08-14](../../til/2026/08/2026-08-14.md)
- 관련 강의자료: [1장 2강: 내적과 코사인 유사도](../../materials/private/kant-basic-math/01-02_내적과_코사인_유사도.pdf)
- 관련 개념: [벡터 L2 정규화](../../knowledge/math/vector-l2-normalization.md)
- 난이도: Core
- 상태: 완료


## 왜 지금 이 실습을 하는가

- TIL에서 확인된 이해: 내적에는 크기와 방향이 함께 반영되고, 코사인 유사도는 정규화한 벡터의 내적이며, 영벡터에는 정의되지 않는다고 설명했다. 또한 코사인 유사도와 유클리드 거리가 서로 다른 것을 측정한다고 해석했다.
- 이번에 확인할 부족한 부분: 여러 후보 벡터를 NumPy로 한 번에 비교하고, 후보의 크기만 바꿨을 때 내적과 코사인 유사도의 검색 순위가 어떻게 달라지는지 실행 결과로 해석한 근거는 아직 없다.
- 핵심 질문: 후보 벡터의 크기를 바꾸면 내적 점수와 코사인 유사도 기반 검색 순위는 어떻게 달라지는가?


## 목표와 완료 기준

### 선수 지식

- 벡터의 내적과 L2 노름을 계산할 수 있다.
- `axis=1`이 각 행의 성분 축을 줄인다는 것을 안다.
- 영벡터를 노름으로 나누면 안 된다는 것을 안다.

### 완료 기준

- [x] 실행 전에 내적 순위와 코사인 유사도 순위를 예상했다.
- [x] 한 쌍의 코사인 유사도 함수를 영벡터 정책과 함께 구현했다.
- [x] 여러 후보의 내적 점수와 코사인 유사도를 계산하고 Shape을 설명했다.
- [x] 후보 하나의 크기만 바꾼 전후 순위를 비교하고 이유를 설명했다.
- [x] 코사인 유사도의 한계나 실패 조건 하나를 적었다.


## 실행 전 예상

아래 준비 셀을 실행하기 전에 먼저 적습니다.

- `query`와 각 후보의 내적을 손으로 계산하면 어떤 순서가 될까?
  - 1, 2, -1, 0 이 되므로 1 > 0 > 3 > 2 순
- 코사인 유사도 순서는 내적 순서와 같을까? 다르다면 어느 후보가 달라질까?
  - 다르다. 각 후보 모두 크기가 동일하고 영벡터가 아니라면 같겠지만 이 조건이 다른 1,3 index 후보들이 달라진다.
- `비스듬한_큰_벡터`를 양수 10배 하면 내적과 코사인 유사도는 각각 어떻게 변할까?
  - 코사인 유사도는 동일하고, 내적만 10배 된다.
- `영벡터`의 코사인 유사도를 그대로 계산하면 어떤 문제가 생길까?
  - 0으로 나누는 계산이 생겨 오류가 발생한다.
- 그렇게 예상한 이유:
  - 코사인 유사도를 구하기 위해서는 해당 벡터의 크기를 나눠야 하는데, 거기서 앞서 말한 오류가 발생한다.
- 비교할 때 같게 유지할 조건: 방향은 그대로 두고 한 후보의 크기만 바꾼다.


In [1]:
# 준비: 데이터를 확인하기 전에 위 예상부터 작성하세요.
import numpy as np

query = np.array([1.0, 0.0])
candidate_names = np.array([
    "같은_방향",
    "비스듬한_큰_벡터",
    "반대_방향",
    "영벡터",
])
candidates = np.array([
    [1.0, 0.0],
    [2.0, 2.0],
    [-1.0, 0.0],
    [0.0, 0.0],
])

print("query.shape:", query.shape)
print("candidates.shape:", candidates.shape)


query.shape: (2,)
candidates.shape: (4, 2)


## 1. 내적 점수와 Shape 확인

1. `query`와 후보 네 개의 내적을 손으로 먼저 계산합니다.
2. `candidates @ query`의 결과 Shape을 예상합니다.
3. NumPy로 내적 점수를 한 번에 계산합니다.
4. 손계산, 예상 Shape, 실제 결과를 비교합니다.

확인할 Shape:

```text
candidates: (후보 수, 성분 수)
query:      (성분 수,)
dot_scores: (?)
```


In [3]:
# TODO: 여러 후보의 내적 점수를 한 번에 계산하세요.
dot_scores = candidates @ query

# TODO: 후보 이름, 내적 점수, 결과 Shape을 출력하세요.
print(f"candidates: {candidates}")
print(f"dot_scores: {dot_scores}")
print(f"dot_scores.shape: {dot_scores.shape}")


candidates: [[ 1.  0.]
 [ 2.  2.]
 [-1.  0.]
 [ 0.  0.]]
dot_scores: [ 1.  2. -1.  0.]
dot_scores.shape: (4,)


## 2. 영벡터를 고려한 코사인 유사도

두 벡터의 코사인 유사도를 계산하는 `cosine_similarity`를 완성합니다.

요구사항:

1. 두 입력의 L2 노름을 구합니다.
2. 어느 한쪽이 영벡터라면 그대로 나누지 않습니다.
3. 영벡터 처리 정책으로 `np.nan` 반환 또는 명시적인 예외 중 하나를 선택합니다.
4. 선택한 정책의 이유를 코드 주석이나 해석 칸에 적습니다.
5. 영벡터가 아닌 후보들의 결과가 손으로 예상한 방향 관계와 맞는지 확인합니다.


In [22]:
def cosine_similarity(a, b):
    """두 벡터의 코사인 유사도를 계산합니다."""
    # TODO: 두 노름을 계산하세요.
    a_norm = np.linalg.norm(a)
    b_norm = np.linalg.norm(b)

    # TODO: 영벡터 처리 정책을 구현하세요.
    if a_norm == 0 or b_norm == 0:
        return np.nan


    # TODO: 내적을 두 노름의 곱으로 나눈 값을 반환하세요.
    return a @ b / (a_norm * b_norm)


# TODO: query와 각 후보의 코사인 유사도를 계산하세요.
cosine_scores = np.array([
    cosine_similarity(query, candidate)
    for candidate in candidates
])

print(f"cosine_scores: {cosine_scores}")


cosine_scores: [ 1.          0.70710678 -1.                 nan]


## 3. 크기만 바꾼 전후의 검색 순위 비교

`비스듬한_큰_벡터`의 방향은 유지한 채 크기만 양수 10배로 바꿉니다.

1. 원본 배열을 복사해 `scaled_candidates`를 만듭니다.
2. 해당 후보만 10배 합니다.
3. 변경 전후의 내적 점수와 코사인 유사도를 계산합니다.
4. 영벡터는 선택한 정책에 따라 순위 비교에서 명시적으로 제외합니다.
5. 내적 순위와 코사인 유사도 순위가 각각 바뀌었는지 설명합니다.

여러 후보의 코사인 유사도를 벡터화하고 싶다면 후보별 노름의 Shape도 먼저 예상합니다.

```text
candidates:      (4, 2)
candidate_norms: (?)
```


In [24]:
scaled_candidates = candidates.copy()

# TODO: '비스듬한_큰_벡터'의 크기만 양수 10배 하세요.
scaled_candidates[1] *= 10

# TODO: 변경 전후의 내적 점수를 계산하세요.
scaled_dot_scores = scaled_candidates @ query

# TODO: 변경 전후의 코사인 유사도를 계산하세요.
scaled_cosine_scores = np.array([
    cosine_similarity(query, candidate)
    for candidate in scaled_candidates
])

# TODO: 영벡터를 제외하고 두 기준의 순위를 비교하세요.
print(f"dot_scores: {dot_scores}")
print(f"scaled_dot_scores: {scaled_dot_scores}")

print(f"cosine_scores: {cosine_scores}")
print(f"scaled_cosine_scores: {scaled_cosine_scores}")


dot_scores: [ 1.  2. -1.  0.]
scaled_dot_scores: [ 1. 20. -1.  0.]
cosine_scores: [ 1.          0.70710678 -1.                 nan]
scaled_cosine_scores: [ 1.          0.70710678 -1.                 nan]


## 막혔을 때 단계별 힌트

<details>
<summary>힌트 1: 여러 후보의 내적</summary>

`candidates`의 각 행과 `query`를 내적하려면 행렬곱 연산자 `@`를 사용할 수 있습니다. 먼저 두 피연산자의 Shape에서 맞닿는 성분 축을 확인하세요.
</details>

<details>
<summary>힌트 2: 한 쌍의 코사인 유사도</summary>

분자는 내적이고 분모는 두 L2 노름의 곱입니다. 나누기 전에 두 노름 중 0이 있는지 확인하세요.
</details>

<details>
<summary>힌트 3: 후보별 노름의 Shape</summary>

후보 하나가 배열의 한 행이라면 성분 축은 `axis=1`입니다. 나눗셈에 필요한 Shape을 생각하며 `keepdims=True`가 필요한지도 판단하세요.
</details>

<details>
<summary>힌트 4: 영벡터와 순위</summary>

영벡터에 임의의 유사도 값을 부여하면 순위 의미가 달라질 수 있습니다. 유효하지 않은 후보임을 나타낸 뒤 순위 계산에서 제외하는 방법을 생각해보세요.
</details>


## 결과 해석과 마무리

- 실제 내적 점수와 코사인 유사도: 코드 참고
- 예상과 같거나 달랐던 부분: 예상과 모두 동일
- 후보의 크기를 10배 했을 때 내적 점수가 변한 이유: 내적은 크기의 영향을 받으므로
- 후보의 크기를 10배 했을 때 코사인 유사도가 변했는지와 그 이유: 변하지 않음.(단, 부동소수점 오차 내의 범위) 코사인 유사도는 크기로 결국 나누기 때문에 방향성만 보기 때문
- 내적 기반 순위와 코사인 유사도 기반 순위의 차이:
- 영벡터 처리 정책과 그 이유: nan처리, 내적 구할 때 크기로 나누는데 0은 나눌 수 없기 때문
- 이 실험이 보여주는 코사인 유사도의 한계 또는 실패 조건: 잘 모르겠음
- 추가로 확인할 점(선택, 없으면 생략):
